# PyTorch in Practice: Tensors, Initialization, Optimizers, Schedules, Early Stopping

*A one-hour notebook for a guided session, written so that it can also be read alone afterwards.*

You already know the basic training loop: forward pass, loss, `loss.backward()`, `optimizer.step()`. This notebook first sharpens two tensor skills you will use constantly, then covers the choices that decide whether that loop trains **well**. Wherever possible it measures instead of asserting, including the cases where a popular technique turns out to make no difference on our small problem.

| § | Topic | planned min |
| --- | --- | --- |
| 0 | Setup: data and two training helpers | 3 |
| **A** | **Tensor skills** | |
| 1 | Tensor internals: storage, strides, views | 6 |
| 2 | `einsum`: index notation as code | 4 |
| **B** | **Training well** | |
| 3 | Weight initialization | 8 |
| 4 | Optimizers and the learning rate | 8 |
| 5 | Learning-rate schedulers | 11 |
| 6 | Weight decay and gradient clipping | 4 |
| 7 | Early stopping, and the effect of more data | 13 |
| | Summary | 3 |

**Requirements.** Python ≥ 3.10, PyTorch 2.x with torchvision, and matplotlib (`pip install torch torchvision matplotlib`; preinstalled on Colab). The notebook was tested end to end with **PyTorch 2.15.0** (released October 2026) on a CPU. A GPU is used automatically if present but was not part of the test.

**Hardware.** All cells together run in about two minutes on a single CPU core. The setup downloads Fashion-MNIST (about 30 MB) once.

**How to use it.** Run all cells first, then read from the top. ⚠️ marks a common pitfall and 💡 a practical tip. Numbers quoted in the text come from our CPU test run; yours may differ slightly with hardware and library versions.

---
# 0. Setup
The first cell imports the libraries and picks a device.

In [ ]:
import copy, math

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import lr_scheduler

torch.manual_seed(0)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"PyTorch {torch.__version__} | device: {device}")

The second cell prepares **Fashion-MNIST**: 28×28 grayscale images of clothing in 10 classes. To keep every run fast and make overfitting visible quickly, we deliberately train on only **10,000 of the 60,000** official training images and validate on 5,000 *other* training images. The official 10,000 test images are used only once, at the very end of §7. The normalization statistics come from the training split only.

It also defines the two helpers used by every experiment:
- `train_one_epoch`: the training loop you know, with one optional extra explained in §5 (a scheduler that is stepped after every batch);
- `evaluate`: mean loss and accuracy in `eval` mode and without autograd.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets

# torchvision tries its own mirror first; the dataset's official GitHub repository is added as a fallback.
GITHUB_MIRROR = "https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/master/data/fashion/"
if GITHUB_MIRROR not in datasets.FashionMNIST.mirrors:
    datasets.FashionMNIST.mirrors = [*datasets.FashionMNIST.mirrors, GITHUB_MIRROR]
raw_train = datasets.FashionMNIST("./data", train=True, download=True)
raw_test = datasets.FashionMNIST("./data", train=False, download=True)

perm = torch.randperm(len(raw_train), generator=torch.Generator().manual_seed(0))
train_idx, val_idx = perm[:10_000], perm[10_000:15_000]       # disjoint subsets of the official training set

def as_vectors(images):
    return images.float().div(255).flatten(1)                  # uint8 (N, 28, 28) -> float (N, 784) in [0, 1]

x_train, y_train = as_vectors(raw_train.data[train_idx]), raw_train.targets[train_idx]
x_val, y_val = as_vectors(raw_train.data[val_idx]), raw_train.targets[val_idx]
x_test, y_test = as_vectors(raw_test.data), raw_test.targets
mean, std = x_train.mean(), x_train.std()                      # statistics of the TRAINING split only
x_train, x_val, x_test = [(x - mean) / std for x in (x_train, x_val, x_test)]

def make_train_loader(seed=0, batch_size=128):
    return DataLoader(TensorDataset(x_train, y_train), batch_size=batch_size, shuffle=True,
                      generator=torch.Generator().manual_seed(seed))
val_loader = DataLoader(TensorDataset(x_val, y_val), batch_size=1000)
test_loader = DataLoader(TensorDataset(x_test, y_test), batch_size=1000)

def train_one_epoch(model, loader, optimizer, scheduler=None):
    # One pass over the training data; returns the mean training loss.
    model.train()
    total_loss, n = 0.0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        loss = F.cross_entropy(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if scheduler is not None:                              # a scheduler passed here is stepped after EVERY batch (§5)
            scheduler.step()
        total_loss += loss.item() * len(y)
        n += len(y)
    return total_loss / n

@torch.no_grad()
def evaluate(model, loader):
    # Mean cross-entropy and accuracy, in eval mode and without autograd.
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        total_loss += F.cross_entropy(logits, y, reduction="sum").item()
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n

print(f"train {len(x_train):,} | val {len(x_val):,} | test {len(x_test):,} | classes: {raw_train.classes}")

---
# Part A · Tensor skills

# 1. Tensor internals: storage, strides and views

A tensor's numbers live in a flat, one-dimensional block of memory, the **storage**. The tensor object itself is only a *description* of how to read that storage: a starting position (`storage_offset()`), a `shape`, and a `stride()`, the number of storage positions to jump for one step along each dimension. Element `[i, j]` lives at storage position

$$\text{offset} + i\cdot\text{stride}[0] + j\cdot\text{stride}[1].$$

Many operations do not touch the numbers at all. They return a new description of the *same* storage, called a **view**. The cell below shows this for a 3×4 tensor: slicing changes the offset and the strides, transposing swaps the strides, and only indexing with a list makes a copy.

In [ ]:
x = torch.arange(12).reshape(3, 4)
print(x, "\n")

def describe(name, t):
    shares = t.untyped_storage().data_ptr() == x.untyped_storage().data_ptr()
    print(f"{name:10s} shape={str(tuple(t.shape)):7s} stride={str(t.stride()):7s} "
          f"offset={t.storage_offset()}  shares memory with x: {shares}")

describe("x", x)                  # element [i, j] is at storage position 0 + 4*i + 1*j
describe("x[1]", x[1])            # row 1: start at position 4, step 1
describe("x[:, 2]", x[:, 2])      # column 2: start at position 2, step 4 (one whole row)
describe("x[::2]", x[::2])        # every other row: the row stride doubles to 8
describe("x.T", x.T)              # transpose: the two strides swap, nothing is moved
describe("x[[0, 2]]", x[[0, 2]])  # indexing with a list or tensor: a copy with its own storage

| Returns a view (shares memory) | Returns a copy (new memory) |
| --- | --- |
| basic slicing `x[1]`, `x[:, 2]`, `x[::2]`; `view`; `transpose`, `permute`, `.T`; `expand`; `squeeze`, `unsqueeze`; `detach` | indexing with a list, an index tensor or a boolean mask, `x[[0, 2]]`, `x[x > 3]`; `clone`; `repeat`; `reshape` *when a view is impossible* |

Views cost nothing to create, but **writing through a view changes the original**. `clone()` gives you independent memory:

In [ ]:
x = torch.zeros(3, 4)
col = x[:, 2]           # a view
col += 7                # an in-place write through the view ...
print(x)                # ... changes x

safe = x[:, 2].clone()  # clone() = a real copy with its own storage
safe += 100
print("x[:, 2] after writing to the clone:", x[:, 2])

## 1.1 Contiguity: why `view` sometimes fails
A tensor is **contiguous** if reading its elements in the usual order (last index fastest) walks through the storage in consecutive positions. A transposed tensor is not: its elements are still stored in the order of the original.

`view` promises never to copy, so it raises an error when the requested shape cannot be described with strides over the existing memory. `reshape` returns a view when that is possible and silently makes a copy otherwise. Use `reshape` unless you specifically need the no-copy guarantee.

In [ ]:
x = torch.arange(6).reshape(2, 3)
xt = x.T
print("x :", x.tolist(), "| stride", x.stride(), "| contiguous:", x.is_contiguous())
print("xt:", xt.tolist(), "| stride", xt.stride(), "| contiguous:", xt.is_contiguous())

try:
    xt.view(6)
except RuntimeError as e:
    print("xt.view(6) failed:", str(e)[:70], "...")
print("xt.reshape(6) works:", xt.reshape(6).tolist(), "| it had to copy:", xt.reshape(6).data_ptr() != x.data_ptr())
print("x.reshape(6) is a view of x (no copy):", x.reshape(6).data_ptr() == x.data_ptr())

Two consequences you will meet in practice:

- ⚠️ **A "snapshot" that keeps changing.** `model.state_dict()` returns tensors that share storage with the live parameters. If you keep it as a "backup" and continue training, the backup changes too. Copy it instead (`clone()` or `copy.deepcopy`). §7 depends on this.
- ⚠️ **In-place writes and autograd.** Autograd keeps some tensors for the backward pass. If you overwrite one of them in place (also through a view), `backward()` stops with the error "one of the variables needed for gradient computation has been modified by an inplace operation". The fix is to write into a `clone()`.

**Takeaway.** A tensor = storage + offset + shape + strides. Views are free and they alias the original; `view` needs a compatible layout, `reshape` copies when it must, and `clone` always gives independent memory.

---
# 2. `einsum`: index notation as code

Papers write formulas such as $y_{ni} = \sum_j x_{nj}W_{ij}$. `torch.einsum` lets you type them almost literally: `y = torch.einsum("nj,ij->ni", x, W)`. The string before `->` labels the dimensions of each input, the string after it those of the output.

**Rules**
1. Each letter labels one dimension of one operand.
2. The same letter in *different* operands means those dimensions are aligned and **multiplied** element by element. Their sizes must be equal (a size of 1 is broadcast).
3. A letter that does **not** appear after `->` is **summed** over.
4. The output dimensions come in the order written after `->`.
5. The same letter *twice in one operand* takes the diagonal: `"ii->"` is the trace.
6. `...` stands for any number of leading dimensions.

The cell writes nine common operations both ways, with `einsum` and in plain PyTorch, and checks that they agree. The last one is the heart of an attention layer: for each item in the batch, the dot product of every query vector with every key vector.

In [ ]:
A, B = torch.randn(3, 3), torch.randn(3, 3)
u, v = torch.randn(3), torch.randn(3)
X, Y = torch.randn(5, 4), torch.randn(5, 4)
lin = nn.Linear(4, 2, bias=False)
P, Q = torch.randn(2, 6, 3, 4), torch.randn(2, 6, 4, 5)
Qy, Ky = torch.randn(2, 7, 8), torch.randn(2, 9, 8)      # 2 sequences; 7 queries and 9 keys of dimension 8

checks = {
    # name:               (einsum version, plain PyTorch version)
    "row sums":           (torch.einsum("ij->i", A), A.sum(dim=1)),
    "trace":              (torch.einsum("ii->", A), A.trace()),
    "dot":                (torch.einsum("i,i->", u, v), u @ v),
    "outer":              (torch.einsum("i,j->ij", u, v), torch.outer(u, v)),
    "matrix product":     (torch.einsum("ij,jk->ik", A, B), A @ B),
    "linear layer":       (torch.einsum("nj,ij->ni", X, lin.weight), lin(X)),
    "row-wise dots":      (torch.einsum("nd,nd->n", X, Y), (X * Y).sum(dim=1)),
    "batched product":    (torch.einsum("...ij,...jk->...ik", P, Q), P @ Q),
    "queries x keys":     (torch.einsum("bqd,bkd->bqk", Qy, Ky), Qy @ Ky.transpose(1, 2)),
}
for name, (a, b) in checks.items():
    print(f"{name:16s} output shape {str(tuple(a.shape)):13s} agrees: {torch.allclose(a, b, atol=1e-5)}")

💡 With two operands, `einsum` is carried out as a (batched) matrix multiplication, so its speed is comparable to `@`; use whichever is easier to read.

**Takeaway.** Label every dimension with a letter; letters shared between operands are multiplied, and letters missing after `->` are summed.

---
# Part B · Training well

# 3. Weight initialization

## 3.1 Why the initial scale matters
A layer computes $z = Wa + b$. In a deep network the size of the signal is multiplied by roughly the same factor at **every** layer. A factor below 1 makes the signal **vanish**, and with it the gradients that reach the early layers; a factor above 1 makes them **explode**. Initialization chooses the weight variance so that this factor is close to 1 *at the start of training*.

For a unit with $n_\text{in}$ inputs and independent, zero-mean weights (ignoring the bias),
$$\operatorname{Var}(z_i) = n_\text{in}\,\operatorname{Var}(w)\,\mathbb{E}[a^2].$$
Keeping this constant from layer to layer gives the two classic rules:

| Initialization | Weight variance | PyTorch call | Typical use |
| --- | --- | --- | --- |
| **He / Kaiming** | $2/n_\text{in}$. The factor 2 compensates for ReLU setting half of its inputs to zero. | `nn.init.kaiming_normal_(w, nonlinearity="relu")` | hidden layers with ReLU-type activations |
| **Xavier / Glorot** | $2/(n_\text{in}+n_\text{out})$, a compromise between the forward signal and the backward gradient | `nn.init.xavier_uniform_(w)` | tanh, sigmoid |
| PyTorch default for `nn.Linear`, `nn.Conv*d` | $1/(3n_\text{in})$ (uniform in $\pm 1/\sqrt{n_\text{in}}$), six times smaller than He | automatic | see §3.2 |

For a convolution, $n_\text{in} = C_\text{in}\cdot k_h\cdot k_w$; PyTorch computes it for you.

## 3.2 Seeing it on a small synthetic problem
To watch the effect in a real training run that takes seconds, we generate a small two-class problem: 600 points on two interleaved spirals. No straight line separates the classes, so a network needs its hidden layers to solve it.

In [ ]:
def make_spirals(n_per_class=300, noise=0.1, seed=0):
    # Two interleaved spiral arms in the plane; the class of a point is the arm it lies on.
    g = torch.Generator().manual_seed(seed)
    angle = 0.5 + 3 * math.pi * torch.rand(n_per_class, generator=g)                    # 1.5 turns
    arm = torch.stack([angle * torch.cos(angle), angle * torch.sin(angle)], dim=1) / math.pi
    x = torch.cat([arm, -arm]) + noise * torch.randn(2 * n_per_class, 2, generator=g)   # 2nd arm = 1st arm rotated by 180°
    y = torch.cat([torch.zeros(n_per_class), torch.ones(n_per_class)]).long()
    return x, y

x_spiral, y_spiral = make_spirals()
print("x_spiral:", tuple(x_spiral.shape), "| y_spiral:", tuple(y_spiral.shape), "| points per class:", y_spiral.bincount().tolist())

plt.figure(figsize=(3.6, 3.6))
plt.scatter(x_spiral[:, 0], x_spiral[:, 1], c=y_spiral, cmap="coolwarm", s=8)
plt.title("two spirals: 600 points, 2 classes", fontsize=10); plt.gca().set_aspect("equal"); plt.tight_layout(); plt.show()

We train small ReLU networks (width 32, no normalization layers) on these points: a **shallow** one with 2 hidden layers and a **deep** one with 8, each once with PyTorch's **default** initialization (too small for ReLU, §3.1) and once with **He** initialization of the hidden-layer weights. For the deep network we add a third version that is deliberately **too large**: He weights multiplied by 2, i.e. four times the He variance. Everything else is identical: SGD with momentum, the same learning rate, 400 steps on all 600 points.

**How to apply an initialization.** Write a function that initializes *one* module, and let `model.apply(fn)` call it on every submodule. The `nn.init.*_` functions work **in place** (trailing underscore) and are not recorded by autograd.

For each network the cell reports, before and after training:
- the **loss**. With two balanced classes, a uniform guess gives $\ln 2 \approx 0.69$;
- the **output spread** (before training only): the standard deviation, across the 600 inputs, of the difference between the two logits. A value near zero means the untrained network gives almost the *same* output for every input, so the input signal did not survive the trip through the layers; a very large value means the signal was amplified on the way;
- the **accuracy** on the training points. Before training it is the result of an untrained network's guesses, so anything from about 50 % to 60 % is chance. (No validation set here: the question is only whether the network can learn at all.)

In [ ]:
def spiral_mlp(depth, width=32):
    layers, n_in = [], 2
    for _ in range(depth):
        layers += [nn.Linear(n_in, width), nn.ReLU()]
        n_in = width
    return nn.Sequential(*layers, nn.Linear(width, 2))

def he_init(m):
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, nonlinearity="relu")   # He initialization of the weights; biases keep their default

def loss_and_accuracy(model):
    with torch.no_grad():
        logits = model(x_spiral)
    return F.cross_entropy(logits, y_spiral).item(), (logits.argmax(dim=1) == y_spiral).float().mean().item(), logits

def train_on_spirals(depth, init, optimizer_name="SGD", steps=400, seed=0):
    torch.manual_seed(seed)
    model = spiral_mlp(depth)
    if init != "default":
        model[:-1].apply(he_init)                                # hidden layers only; the output layer keeps its default
    if init == "2 x He":
        with torch.no_grad():
            for layer in model[:-1]:
                if isinstance(layer, nn.Linear):
                    layer.weight.mul_(2)                         # deliberately too large
    loss_before, acc_before, logits = loss_and_accuracy(model)
    spread = (logits[:, 1] - logits[:, 0]).std().item()
    if optimizer_name == "SGD":
        optimizer = torch.optim.SGD(model.parameters(), lr=0.03, momentum=0.9)
    else:
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    losses = []
    for _ in range(steps):                                       # full batch: all 600 points in every step
        loss = F.cross_entropy(model(x_spiral), y_spiral)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    loss_after, acc_after, _ = loss_and_accuracy(model)
    return loss_before, acc_before, spread, losses, loss_after, acc_after

runs = [(2, "default", "SGD"), (2, "He", "SGD"),
        (8, "default", "SGD"), (8, "He", "SGD"), (8, "2 x He", "SGD"), (8, "default", "Adam")]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.3), sharey=True)
print("hidden |         |           |        before training         |  after training")
print("layers | init    | optimizer |    loss  accuracy       spread |    loss  accuracy")
for depth, init, optimizer_name in runs:
    loss_before, acc_before, spread, losses, loss_after, acc_after = train_on_spirals(depth, init, optimizer_name)
    print(f"{depth:6d} | {init:7s} | {optimizer_name:9s} | {loss_before:7.3f} {acc_before:9.1%} {spread:12.4f} | {loss_after:7.4f} {acc_after:9.1%}")
    if all(math.isfinite(v) for v in losses):                    # a run that ended in NaN cannot be drawn
        axes[0 if depth == 2 else 1].semilogy(losses, ls="--" if optimizer_name == "Adam" else "-",
                                              label=f"{init} init, {optimizer_name}")
axes[0].set(title="shallow: 2 hidden layers", xlabel="training step", ylabel="training loss (log scale)")
axes[1].set(title="deep: 8 hidden layers", xlabel="training step")
for ax in axes:
    ax.axhline(math.log(2), color="gray", lw=0.8, ls=":"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

The dotted line marks $\ln 2$, the loss of a network that has learned nothing.

- **Shallow network:** both initializations train, and the two curves almost coincide. Two hidden layers are too few for a wrong scale to compound. (He starts somewhat above $\ln 2$ because its larger weights produce larger initial logits.)
- **Deep network, default initialization, SGD (vanishing):** the output spread before training is close to zero. Each of the eight layers shrinks the input-dependent signal, so the untrained network gives practically the same output for every point and puts all points in one class (50 % accuracy). The gradients that flow back to the early layers are just as small. After 400 steps the loss is still $\ln 2$. The accuracy has crept above 50 % only because the almost-constant output now leans very slightly one way or the other; nothing useful has been learned.
- **Deep network, He initialization, SGD:** the output spread is of the same order as in the shallow network, and the same optimizer with the same learning rate now fits the spirals.
- **Deep network, 2 × He, SGD (exploding):** each layer now amplifies the signal. The output spread and the loss before training are far too large, the first gradient steps are enormous, and training ends in `nan` (which is why this run has no curve).
- **Deep network, default initialization, Adam** (dashed): the *same badly scaled network* does train. Adam, introduced in §4.3, divides every gradient by its typical size, which compensates for gradients that are uniformly tiny. So the optimizer can hide a poor initialization. A correct initialization is still the standard, because it does not depend on the optimizer to repair it.

All of this is one seed; change `seed` in `train_on_spirals` to see how much the picture depends on it.

**Normalization layers** (BatchNorm, LayerNorm) and **residual connections** are the other common remedy: they keep the signal scale under control by construction and make networks much less sensitive to the initial weights.

## 3.3 The network for the rest of the notebook
All experiments of §4–7 use one MLP for Fashion-MNIST, 784 → 512 → 512 → 10, with He initialization for the hidden weights. With two hidden layers it is in the "shallow" regime above, so the choice is not critical here; we use He because it is the right habit for ReLU networks.

A sanity check worth doing for every new model: with 10 balanced classes, the loss **before training** should be close to $\ln 10 \approx 2.30$. The cell also shows what happens if the *output* layer is given He initialization too.

In [ ]:
def make_mlp(hidden=512):
    model = nn.Sequential(nn.Linear(784, hidden), nn.ReLU(),
                          nn.Linear(hidden, hidden), nn.ReLU(),
                          nn.Linear(hidden, 10))
    model[:-1].apply(he_init)                 # He for the hidden layers; the output layer keeps the default
    return model.to(device)

torch.manual_seed(0)
model = make_mlp()
print(f"std of the first-layer weights: {model[0].weight.std():.4f}  (He target sqrt(2/784) = {math.sqrt(2 / 784):.4f})")
print(f"loss before training, He hidden + default output layer: {evaluate(model, val_loader)[0]:.2f}   (ln 10 = {math.log(10):.2f})")
torch.manual_seed(0)
model_he_everywhere = make_mlp().apply(he_init)
print(f"loss before training, He on every layer:                {evaluate(model_he_everywhere, val_loader)[0]:.2f}")

With He initialization on the output layer the initial logits are larger, and the loss before training is clearly above $\ln 10$: the untrained model already prefers some classes before it has seen any data. Leave the output layer at its default, smaller scale.

**Takeaway.** Initialization decides whether a *deep* network *without normalization* can train: use He for ReLU-type hidden layers and Xavier for tanh/sigmoid, and keep the output layer small. For shallow or normalized networks the PyTorch default is fine. Always check that the loss before training is near $\ln(\text{number of classes})$.

---
# 4. Optimizers

## 4.1 What an optimizer is
An optimizer updates the parameters from their gradients. It holds **`param_groups`**, a list of dicts with parameters and their settings (`lr`, `weight_decay`, …; used in §6), and **`state`**, its memory per parameter, such as momentum buffers.

Below, $g$ is a parameter's gradient and $\eta$ the learning rate (`lr`).

## 4.2 SGD with momentum
PyTorch keeps a running **sum** of past gradients, the velocity $v$:
$$v \leftarrow \mu v + g, \qquad p \leftarrow p - \eta v.$$
Where the gradient keeps its sign, the terms add up (faster progress); where it oscillates, they cancel. For a constant gradient $v \to g/(1-\mu)$: with $\mu = 0.9$ the steps are about **10× larger** than those of plain SGD with the same `lr`.

`torch.optim.SGD(params, lr=..., momentum=0.9, weight_decay=...)`. The default `momentum` is 0 (plain SGD); 0.9 is the usual starting value. The `lr` **must be tuned** (§4.6).

## 4.3 Adam
Adam keeps exponential moving averages of the gradient ($m$) and of its square ($v$), corrects their bias toward zero in the first steps, and divides one by the square root of the other:
$$m \leftarrow \beta_1 m + (1-\beta_1)g,\qquad v \leftarrow \beta_2 v + (1-\beta_2)g^2,\qquad \hat m = \frac{m}{1-\beta_1^t},\quad \hat v = \frac{v}{1-\beta_2^t},$$
$$p \leftarrow p - \eta\,\frac{\hat m}{\sqrt{\hat v}+\varepsilon}.$$
The ratio $\hat m/\sqrt{\hat v}$ is about ±1 when a parameter's gradient is consistent, **whatever its scale**, and smaller when the gradient is noisy. Each parameter therefore typically moves by about $\eta$ per step or less. Because every parameter is scaled separately, Adam copes well with parameters whose gradients differ by orders of magnitude, and its default `lr` works reasonably often.

`torch.optim.Adam(params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)`

## 4.4 AdamW
AdamW is Adam with **decoupled weight decay**: at every step each weight is additionally shrunk a little, $p \leftarrow p - \eta\lambda p$, independently of the gradient scaling (details in §6). It is the usual first choice today and the standard optimizer for transformers.

`torch.optim.AdamW(params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01)`

⚠️ The default `weight_decay` is 0.01 in `AdamW` but 0 in `Adam`. And `Adam(weight_decay=λ)` is *not* the same thing as `AdamW(weight_decay=λ)` (§6).

## 4.5 Which one, with which settings?

| Optimizer | Typical starting values | When |
| --- | --- | --- |
| **AdamW** | `lr=1e-3` for small MLPs/CNNs, `3e-4` or lower for transformers; `weight_decay=0.01–0.1` | the default first choice |
| **SGD + momentum** | `momentum=0.9`, `lr≈0.1` at batch size 256, `weight_decay=1e-4–5e-4`, always with a schedule (§5) | CNNs with established recipes, where it can match or beat AdamW |
| **Adam** | `lr=1e-3` | when you want no weight decay (identical to AdamW with `weight_decay=0`) |

⚠️ **Comparing optimizers.** Give each one the same tuning effort, at the very least its own learning-rate sweep (§4.6). Their best learning rates differ by orders of magnitude (compare 0.1 and 1e-3 above), so a comparison at one shared value mostly measures which optimizer that value happens to suit.

💡 SGD with momentum stores one extra tensor per parameter, Adam and AdamW two; the cell shows this state. `torch.optim` contains more optimizers. The newest, `torch.optim.Muon` (since PyTorch 2.9), is used for the weight matrices of some large language models, with AdamW for the remaining parameters. AdamW remains the standard default.

In [ ]:
lin = nn.Linear(3, 2)
for opt in [torch.optim.SGD(lin.parameters(), lr=0.1, momentum=0.9), torch.optim.AdamW(lin.parameters(), lr=1e-3)]:
    lin.zero_grad(); lin(torch.randn(8, 3)).pow(2).mean().backward(); opt.step()
    print(f"{type(opt).__name__:6s} state for the weight tensor after one step: {list(opt.state[lin.weight].keys())}")

## 4.6 Choosing the learning rate: a short sweep
The learning rate is the most important hyper-parameter, so tune it, at least roughly:
1. Try values on a **logarithmic grid**, about 3× apart (here 1e-4 … 1e-2).
2. Train each for a **short** time (here 2 epochs) and compare the **validation** loss, never the test loss.
3. Take the best value and confirm it with a full run. If the best value lies at the edge of the grid, extend the grid.

The best value after a short run is not guaranteed to be the best for a long run; the sweep finds the right **order of magnitude**.

In [ ]:
sweep_lrs = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]
sweep_val = []
for lr in sweep_lrs:
    torch.manual_seed(0)
    model = make_mlp()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    loader = make_train_loader(seed=0)
    for _ in range(2):
        train_one_epoch(model, loader, optimizer)
    val_loss, val_acc = evaluate(model, val_loader)
    sweep_val.append(val_loss)
    print(f"lr {lr:.0e}: validation loss {val_loss:.4f}, accuracy {val_acc:.4f}")

plt.figure(figsize=(5.5, 3.2))
plt.semilogx(sweep_lrs, sweep_val, "o-")
plt.xlabel("learning rate (log scale)"); plt.ylabel("validation loss after 2 epochs")
plt.title("AdamW learning-rate sweep"); plt.tight_layout(); plt.show()
print("best learning rate in the sweep:", sweep_lrs[sweep_val.index(min(sweep_val))])

The curve is U-shaped: with too small a learning rate training is slow, with too large a one it becomes unstable. For this MLP and AdamW the minimum was at `1e-3` in our run, so that value is used below.

**Takeaway.** Start with AdamW (`lr=1e-3`, `weight_decay=0.01`) and tune the learning rate with a short log-scale sweep on the validation set. Use SGD with momentum and a schedule when you follow an established recipe for CNNs. Compare optimizers only after tuning them equally.

---
# 5. Learning-rate schedulers

## 5.1 Why and how
Early in training, large steps make fast progress; late in training, small steps let the weights settle, so most recipes **decay** the learning rate. Many also **warm up**: they start with a small learning rate for the first steps, because the first updates, computed from random weights and (for Adam) from averages over very few gradients, can be large and destabilizing.

A scheduler is an object that wraps an optimizer. Each call of `scheduler.step()` advances the schedule by one and writes the new learning rate into the optimizer.

**Rules that prevent most scheduler bugs**
1. Call `scheduler.step()` **after** `optimizer.step()`.
2. Decide whether you step **per epoch or per batch**, and size the schedule in the **same unit**.
3. `ReduceLROnPlateau` is the exception: step it once per epoch **with the monitored value**, `scheduler.step(val_loss)` (§5.4).
4. Print or plot the learning rate. Most scheduler bugs are obvious as soon as you look at it.

## 5.2 A minimal, correct example
Before comparing schedulers, here is one used correctly in a complete training loop. We train a small network on the spiral data of §3.2 in mini-batches for 20 epochs with `CosineAnnealingLR(optimizer, T_max=EPOCHS)`: the learning rate falls smoothly from its starting value to 0 over exactly `T_max` scheduler steps. We step it **once per epoch**, so `T_max` is given in epochs. The four numbered comments are the whole recipe.

In [ ]:
EPOCHS = 20
torch.manual_seed(0)
spiral_loader = DataLoader(TensorDataset(x_spiral, y_spiral), batch_size=50, shuffle=True,
                           generator=torch.Generator().manual_seed(0))               # 600 points -> 12 batches per epoch
model = spiral_mlp(depth=2)
model[:-1].apply(he_init)

optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)
scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)   # 1. create the scheduler AFTER the optimizer it wraps

for epoch in range(1, EPOCHS + 1):
    lr_used = scheduler.get_last_lr()[0]                              # 2. read the learning rate in effect (to log it)
    total_loss, correct = 0.0, 0
    for x, y in spiral_loader:
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()                                              # 3. first update the weights ...
        total_loss += loss.item() * len(y)
        correct += (logits.argmax(dim=1) == y).sum().item()
    scheduler.step()                                                  # 4. ... then advance the schedule: here once per EPOCH
    if epoch % 2 == 0:
        print(f"epoch {epoch:2d} | lr used {lr_used:.4f} | training loss {total_loss / len(x_spiral):.3f} | accuracy {correct / len(x_spiral):.1%}")

The printed learning rate follows the cosine from 0.1 down toward 0, while the loss falls and the accuracy rises.

What to take from this example:
- **Order.** The scheduler is created after the optimizer, and `scheduler.step()` comes after `optimizer.step()`. If you call it first, PyTorch warns you, and the first value of the schedule is skipped.
- **Unit.** The scheduler does not know what an epoch is; it counts calls of `scheduler.step()`. Because line 4 sits *outside* the batch loop, one call is one epoch, and `T_max=EPOCHS` is right. To step per batch instead, move that line *inside* the batch loop and give the length in batches: `T_max=EPOCHS * len(spiral_loader)`. The `train_one_epoch` helper of §0 steps a scheduler per batch, and §5.6 uses it that way.
- **The schedule must match the run.** A schedule that reaches zero long before training ends stops the learning; one that is far longer than the run never decays. Size it to the number of steps you will really take.
- **Reading the learning rate.** `scheduler.get_last_lr()` returns a list with one value per parameter group; `optimizer.param_groups[0]["lr"]` is the same number.

This example shows how to use a scheduler, not that this one helps; whether a schedule helps is measured in §5.6.

## 5.3 The most important schedulers

| Scheduler | What it does | Use it when |
| --- | --- | --- |
| `StepLR` / `MultiStepLR` | multiplies `lr` by `gamma` (e.g. 0.1) every `step_size` steps / at the given `milestones` | reproducing classic recipes (e.g. ResNet with SGD) |
| `CosineAnnealingLR` | smooth decay from `lr` to `eta_min` (default 0) along half a cosine over `T_max` steps | you know the training length; a strong, simple default |
| warm-up + cosine: `SequentialLR` of `LinearLR` and `CosineAnnealingLR` | linear ramp-up over the first steps, then cosine decay | the standard recipe for transformers and for large batches or learning rates, where the first steps are unstable without it |
| `OneCycleLR` | rises to `max_lr`, then anneals far below the starting value; **stepped per batch** | short runs with a fixed number of steps |
| `ReduceLROnPlateau` | multiplies `lr` by `factor` when the validation metric stops improving | unknown training length; pairs naturally with early stopping (§7) |

The first four follow a fixed curve. To compare their shapes, the cell records the learning rate of each over 100 scheduler steps with a base `lr` of 1, using a dummy optimizer and the same call order as above.

In [ ]:
def lr_trace(make_scheduler, steps=100, base_lr=1.0):
    # The learning rate in effect at each step, recorded with a dummy optimizer.
    opt = torch.optim.SGD([torch.zeros(1, requires_grad=True)], lr=base_lr)
    sched = make_scheduler(opt)
    lrs = []
    for _ in range(steps):
        lrs.append(opt.param_groups[0]["lr"])   # lr used for this step
        opt.step()                              # (a real training step would happen here)
        sched.step()                            # AFTER optimizer.step()
    return lrs

S = lr_scheduler
schedules = {
    "MultiStepLR(milestones=[50, 75], gamma=0.1)":
        lambda o: S.MultiStepLR(o, milestones=[50, 75], gamma=0.1),
    "CosineAnnealingLR(T_max=100)":
        lambda o: S.CosineAnnealingLR(o, T_max=100),
    "warm-up (10 steps) + cosine (90 steps)":
        lambda o: S.SequentialLR(o, schedulers=[S.LinearLR(o, start_factor=0.01, total_iters=10),
                                                S.CosineAnnealingLR(o, T_max=90)], milestones=[10]),
    "OneCycleLR(max_lr=1, total_steps=100)":
        lambda o: S.OneCycleLR(o, max_lr=1.0, total_steps=100),
}
fig, axes = plt.subplots(1, 4, figsize=(16, 2.8), sharey=True)
for ax, (name, make) in zip(axes, schedules.items()):
    ax.plot(lr_trace(make)); ax.set_title(name, fontsize=8); ax.set_xlabel("scheduler step")
axes[0].set_ylabel("learning rate"); plt.tight_layout(); plt.show()

Details worth knowing:
- `OneCycleLR` **ignores the optimizer's `lr`**: it starts at `max_lr / 25`, peaks at `max_lr` after 30 % of the steps, and ends far below the starting value (these are its defaults). Calling `step()` more than `total_steps` times raises an error.
- ⚠️ By default `OneCycleLR` also **cycles the momentum** (for Adam: β₁) between 0.85 and 0.95. On `SGD` with `momentum=0` this silently turns momentum on. Pass `cycle_momentum=False` if you do not want it.

## 5.4 `ReduceLROnPlateau` reacts to the validation loss
This scheduler has no fixed curve. With `patience=2` it **tolerates two epochs** without improvement and reduces the learning rate at the **third**; then the count starts again. "Improvement" means, by default, at least 0.01 % below the best value so far (`threshold=1e-4`, `threshold_mode="rel"`). Below it is fed a made-up sequence of validation losses that stalls twice.

In [ ]:
opt = torch.optim.SGD([torch.zeros(1, requires_grad=True)], lr=0.1)
plateau = lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=2)
best = math.inf
for epoch, val_loss in enumerate([1.00, 0.80, 0.70, 0.71, 0.70, 0.72, 0.65, 0.60, 0.61, 0.62, 0.61, 0.60], start=1):
    opt.step()                    # (one epoch of training would happen here)
    plateau.step(val_loss)        # once per epoch, WITH the monitored value
    improved = val_loss < best * (1 - 1e-4)
    best = min(best, val_loss)
    print(f"epoch {epoch:2d} | val loss {val_loss:.2f} | {'improved   ' if improved else 'no progress'} | "
          f"lr for the next epoch: {opt.param_groups[0]['lr']:.3f}")

## 5.5 ⚠️ The most common bug: mixing units
This is rule 2 violated: a cosine schedule sized for 10 *epochs* (`T_max=10`) but stepped after every *batch*, with 50 batches per epoch, next to the correct version.

In [ ]:
wrong = lr_trace(lambda o: S.CosineAnnealingLR(o, T_max=10), steps=10 * 50)
right = lr_trace(lambda o: S.CosineAnnealingLR(o, T_max=10 * 50), steps=10 * 50)
plt.figure(figsize=(8, 2.8))
plt.plot(wrong, label="T_max=10 (epochs), stepped every batch: BUG")
plt.plot(right, label="T_max=500 (batches), stepped every batch: correct")
plt.xlabel("batch"); plt.ylabel("learning rate"); plt.legend(fontsize=8); plt.tight_layout(); plt.show()

The buggy schedule reaches zero after 10 batches, climbs back and repeats: 25 full oscillations instead of one decay. No error and no warning is raised; only a plot of the learning rate reveals it.

## 5.6 Does the schedule matter? A real run
We train the MLP of §3.3 for 10 epochs with AdamW (`lr=1e-3`) in three versions: a **constant** learning rate, **cosine decay**, and **one epoch of linear warm-up followed by cosine decay**. Testing decay with and without warm-up shows which of the two ingredients is responsible for a difference. The schedulers are stepped after every **batch**, so they are sized in batches. Each version runs with **two seeds**, to see whether a difference is real or luck.

In [ ]:
EPOCHS = 10
colors = {"constant": "tab:blue", "cosine": "tab:green", "warm-up + cosine": "tab:orange"}
schedule_runs = {}
for seed in (0, 1):
    for name in colors:
        torch.manual_seed(seed)
        model = make_mlp()
        loader = make_train_loader(seed)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
        warmup, total = len(loader), EPOCHS * len(loader)                     # sized in BATCHES
        if name == "constant":
            scheduler = None
        elif name == "cosine":
            scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=total)
        else:
            scheduler = lr_scheduler.SequentialLR(optimizer, milestones=[warmup], schedulers=[
                lr_scheduler.LinearLR(optimizer, start_factor=0.01, total_iters=warmup),
                lr_scheduler.CosineAnnealingLR(optimizer, T_max=total - warmup)])
        val_curve, lr_curve = [], []
        for epoch in range(EPOCHS):
            lr_curve.append(optimizer.param_groups[0]["lr"])
            train_one_epoch(model, loader, optimizer, scheduler)              # scheduler stepped per batch
            val_curve.append(evaluate(model, val_loader))
        schedule_runs[(name, seed)] = (val_curve, lr_curve)
        print(f"seed {seed} | {name:17s} | final validation loss {val_curve[-1][0]:.4f}, accuracy {val_curve[-1][1]:.4f}")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.4))
for (name, seed), (val_curve, lr_curve) in schedule_runs.items():
    style = dict(color=colors[name], ls="-" if seed == 0 else "--")
    a1.plot(range(1, EPOCHS + 1), [v[0] for v in val_curve], marker="o", ms=3, label=f"{name}, seed {seed}", **style)
    a2.plot(range(1, EPOCHS + 1), lr_curve, marker="o", ms=3, **style)
a1.set(xlabel="epoch", ylabel="validation loss", title="Validation loss"); a1.legend(fontsize=7)
a2.set(xlabel="epoch", ylabel="lr at the start of the epoch", title="Learning rate"); plt.tight_layout(); plt.show()

In our run:
- **Decay helps.** Both decaying schedules ended with a clearly lower validation loss than the constant learning rate, for both seeds (about 0.36 versus 0.40 and 0.44). With the constant rate the validation loss stops improving after two or three epochs and then fluctuates, with a large spike for one seed; the decaying rate lets the weights settle.
- **Warm-up made no measurable difference here.** Cosine with and without warm-up ended within 0.002 of each other (warm-up was marginally behind for both seeds), far too little to matter. This small MLP is stable from the first step, so it has nothing for warm-up to fix. Warm-up earns its place where the first steps are unstable: transformers, very deep networks, large batches and large learning rates.

Two seeds are the bare minimum for such claims; in a report, use more and show the spread.

**How to choose, in one line each**
- Fixed training budget → **cosine**; add a linear **warm-up** for transformers, or whenever the first steps of training are unstable.
- Short run with a fixed number of steps → **OneCycleLR**, stepped per batch.
- Unknown number of epochs, or you use early stopping → **ReduceLROnPlateau**.
- Reproducing a paper → whatever the paper used.

---
# 6. Weight decay and gradient clipping

## 6.1 Weight decay
Weight decay shrinks every weight a little toward zero at each step, $p \leftarrow p - \eta\lambda p$. This discourages large weights and often improves generalization. It is not guaranteed to help; the validation set decides.

- **AdamW:** `AdamW(..., weight_decay=λ)` applies exactly this shrink, with the *current* learning rate $\eta$. Typical λ: 0.01–0.1.
- **SGD:** `SGD(..., weight_decay=λ)` adds $\lambda p$ to the gradient. For plain SGD this is exactly the same shrink; with momentum it is not identical but behaves similarly, and it is what the established recipes use. Typical λ: 1e-4–5e-4.
- ⚠️ **Not `Adam(weight_decay=λ)`.** Adam also adds $\lambda p$ to the gradient, but then divides by $\sqrt{\hat v}$, so weights with large gradients are barely decayed. Use `AdamW`.
- The shrink per step is $\eta\lambda$, so the total effect grows with the learning rate and the number of steps. Retune λ when you change either substantially.
- **Do not decay biases and normalization parameters** (a widespread convention). In typical models these are the 1-D parameters, so two parameter groups selected by `p.ndim` do the job, as in the cell below. (§4–5 passed `model.parameters()` to AdamW, which decays everything with the default 0.01. §7.4 shows how small the total effect of this decay is in our short runs.)

In [ ]:
def param_groups(model, weight_decay):
    # Two groups: weight matrices and kernels are decayed; biases and normalization parameters (1-D) are not.
    decay, no_decay = [], []
    for p in model.parameters():
        if p.requires_grad:
            (no_decay if p.ndim <= 1 else decay).append(p)
    return [{"params": decay, "weight_decay": weight_decay},
            {"params": no_decay, "weight_decay": 0.0}]

demo = nn.Sequential(nn.Linear(784, 256), nn.LayerNorm(256), nn.ReLU(), nn.Linear(256, 10))
opt = torch.optim.AdamW(param_groups(demo, weight_decay=0.05), lr=1e-3)
for i, g in enumerate(opt.param_groups):
    print(f"group {i}: {len(g['params'])} tensors, {sum(p.numel() for p in g['params']):,} values, "
          f"weight_decay={g['weight_decay']}")

## 6.2 Gradient clipping
Occasionally one batch produces a very large gradient, and a single huge step undoes much of the progress or produces `NaN`. **Gradient clipping** prevents this: if the norm of *all* gradients together exceeds `max_norm`, every gradient is multiplied by the same factor so that the total norm equals `max_norm`. The direction is unchanged; only the length is limited.

It is one line, placed **between `loss.backward()` and `optimizer.step()`**:
```python
total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
```
The function returns the norm *before* clipping.

💡 Logging that value is a cheap health check for any training run. A typical choice is `max_norm=1.0`. Clipping is standard for transformers and recurrent networks. Our small MLP trains stably without it, so the training runs in this notebook do not use it; the cell only demonstrates the mechanics on one batch.

In [ ]:
torch.manual_seed(0)
model = make_mlp()
xb, yb = next(iter(make_train_loader()))
F.cross_entropy(model(xb.to(device)), yb.to(device)).backward()

norm_before = nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)      # returns the norm BEFORE clipping
norm_after = torch.sqrt(sum(p.grad.pow(2).sum() for p in model.parameters()))
print(f"total gradient norm before clipping: {norm_before:.3f} | after clipping to max_norm=1.0: {norm_after:.3f}")

---
# 7. Early stopping

## 7.1 The idea
The training loss keeps falling as long as you train. The **validation** loss usually falls, reaches a minimum, and then rises again as the model starts to memorize the training set (overfitting). Early stopping:

1. evaluates on the validation set after every epoch;
2. remembers the **best** weights seen so far;
3. stops when the monitored value has not improved by more than `min_delta` for `patience` epochs in a row;
4. **restores the best weights** before the model is used.

## 7.2 Doing it correctly: a checklist
- **Monitor a validation set, never the test set.** The test set is used once, at the very end.
- **Evaluate in evaluation mode**, `model.eval()` and `torch.no_grad()`, and switch back with `model.train()` (our helpers do both). Dropout and BatchNorm behave differently in the two modes.
- **Copy the best weights** (the snapshot pitfall of §1.1), on the **CPU** and on disk, and **restore them** at the end: when early stopping ends a run, the last epoch is `patience` epochs past the best one.
- **Prefer the validation loss** as the monitored value: it is smoother than accuracy and reacts earlier. Treat NaN as "no improvement".
- **Choose `patience` clearly larger than that of a `ReduceLROnPlateau` scheduler**, so that the learning rate can drop, and the lower rate can show its effect, before training stops. Give both the **same definition of improvement**.
- **Pair it with a suitable schedule.** Early stopping fits a constant learning rate or `ReduceLROnPlateau`. A cosine or OneCycle schedule is sized for the full run, and stopping in the middle leaves it at a high learning rate; with those, train to the end and keep the checkpoint with the best validation loss.

In [ ]:
class EarlyStopping:
    # Stop when the monitored value has not improved by more than `min_delta` for `patience` epochs in a row.
    # mode="min" for losses, "max" for accuracies. Keeps a CPU copy of the best weights and,
    # if `path` is given, also saves them to disk at every improvement.

    def __init__(self, patience=5, min_delta=0.0, mode="min", path=None):
        assert mode in ("min", "max")
        self.patience, self.min_delta, self.mode, self.path = patience, min_delta, mode, path
        self.best = math.inf if mode == "min" else -math.inf
        self.best_epoch, self.best_state, self.bad_epochs = None, None, 0

    def is_improvement(self, value):
        if not math.isfinite(value):                     # NaN or inf never counts as an improvement
            return False
        if self.mode == "min":
            return value < self.best - self.min_delta
        return value > self.best + self.min_delta

    def step(self, value, model, epoch):
        # Call once per epoch with the validation metric. Returns True when training should stop.
        if self.is_improvement(value):
            self.best, self.best_epoch, self.bad_epochs = value, epoch, 0
            # an independent COPY of every tensor (not a reference to the live weights), kept on the CPU
            self.best_state = {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}
            if self.path is not None:
                torch.save({"epoch": epoch, "value": value, "model_state": self.best_state}, self.path)
        else:
            self.bad_epochs += 1
        return self.bad_epochs >= self.patience

    def restore_best(self, model):
        if self.best_state is None:
            raise RuntimeError("No epoch ever improved (e.g. every validation loss was NaN): nothing to restore.")
        model.load_state_dict(self.best_state)           # copies the values into the model's own tensors

A quick test with made-up validation losses and `patience=3`. Note that the word has two meanings: this class stops at the `patience`-th epoch in a row without improvement, whereas `ReduceLROnPlateau` acts one epoch later, at the (`patience` + 1)-th (§5.4).

In [ ]:
stopper = EarlyStopping(patience=3)
dummy = nn.Linear(1, 1)
for epoch, val_loss in enumerate([1.0, 0.8, 0.7, 0.72, 0.69, 0.71, 0.70, 0.73, 0.75], start=1):
    stop = stopper.step(val_loss, dummy, epoch)
    print(f"epoch {epoch}: val loss {val_loss:.2f} | best {stopper.best:.2f} (epoch {stopper.best_epoch}) | "
          f"epochs without improvement: {stopper.bad_epochs}{'  -> STOP' if stop else ''}")
    if stop:
        break

## 7.3 ⚠️ Why the copy matters
This is §1 in action. `state_dict()` hands out tensors that share memory with the live parameters:

In [ ]:
net = nn.Linear(2, 1)
snapshot_wrong = net.state_dict()                   # tensors that share memory with the live parameters
snapshot_right = copy.deepcopy(net.state_dict())    # an independent copy

with torch.no_grad():
    net.weight += 1.0                               # "training continues" and changes the weights

print("wrong snapshot is equal to the CURRENT weights:", torch.equal(snapshot_wrong["weight"], net.weight))
print("right snapshot is equal to the CURRENT weights:", torch.equal(snapshot_right["weight"], net.weight))

The "wrong" snapshot silently followed the training: restoring it would give you the *last* weights, not the best ones, without any error.

## 7.4 Everything together on Fashion-MNIST
A real training run that uses the pieces of this notebook:
- **Model:** the MLP of §3.3, with He initialization for the hidden layers.
- **Optimizer:** **AdamW** with `lr=1e-3` from the sweep in §4.6 and **weight decay** 0.01 on the weight matrices only (§6.1).
- **Schedule:** **`ReduceLROnPlateau`** with `factor=0.5` and `patience=2` (§5.4).
- **Early stopping** with `patience=6` and at most 50 epochs.
- Scheduler and early stopping use the **same definition of improvement**: a validation loss at least 0.001 lower than the best so far.
- The best weights are also saved to disk.

We run it for **three seeds**. The per-epoch log is printed for seed 0 only; each finished model is evaluated on the test set exactly once, after all decisions have been made.

In [ ]:
WEIGHT_DECAY = 0.01

def run_with_early_stopping(seed, verbose=False):
    torch.manual_seed(seed)
    model = make_mlp()
    loader = make_train_loader(seed)
    optimizer = torch.optim.AdamW(param_groups(model, WEIGHT_DECAY), lr=1e-3)
    scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2,
                                               threshold=1e-3, threshold_mode="abs")
    stopper = EarlyStopping(patience=6, min_delta=1e-3, path=f"best_model_seed{seed}.pt")
    history = {"train_loss": [], "val_loss": [], "val_acc": [], "lr": []}
    for epoch in range(1, 51):
        lr = optimizer.param_groups[0]["lr"]
        train_loss = train_one_epoch(model, loader, optimizer)
        val_loss, val_acc = evaluate(model, val_loader)
        scheduler.step(val_loss)                           # plateau scheduler: once per epoch, with the value
        for key, value in zip(history, (train_loss, val_loss, val_acc, lr)):
            history[key].append(value)
        if verbose:
            print(f"epoch {epoch:2d} | lr {lr:.1e} | train loss {train_loss:.4f} | val loss {val_loss:.4f} | "
                  f"val acc {val_acc:.4f}{'  * best so far' if stopper.is_improvement(val_loss) else ''}")
        if stopper.step(val_loss, model, epoch):
            break
    stopper.restore_best(model)                            # keep the BEST epoch, not the last
    return model, stopper, history

results = {}
for seed in (0, 1, 2):
    model, stopper, history = run_with_early_stopping(seed, verbose=(seed == 0))
    test_loss, test_acc = evaluate(model, test_loader)     # the test set: once per finished model
    results[seed] = (stopper, history, test_loss, test_acc)
    print(f"seed {seed}: stopped after epoch {len(history['val_loss']):2d} | restored epoch {stopper.best_epoch:2d} "
          f"(validation loss {stopper.best:.4f}) | test loss {test_loss:.4f}, test accuracy {test_acc:.4f}")

In [ ]:
stopper, history, _, _ = results[0]                        # the plots show seed 0
epochs = range(1, len(history["val_loss"]) + 1)
fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(15, 3.3))
a1.plot(epochs, history["train_loss"], "o-", ms=3, label="train")
a1.plot(epochs, history["val_loss"], "o-", ms=3, label="validation")
a2.plot(epochs, history["val_acc"], "o-", ms=3, color="tab:green")
a3.semilogy(epochs, history["lr"], "o-", ms=3, color="tab:purple")
for ax, title in zip((a1, a2, a3), ("cross-entropy loss (seed 0)", "validation accuracy (seed 0)", "learning rate (seed 0, log scale)")):
    ax.axvline(stopper.best_epoch, color="red", ls="--", lw=1)
    ax.set_title(title); ax.set_xlabel("epoch")
a1.legend(); plt.tight_layout(); plt.show()

steps_per_epoch = len(make_train_loader())
shrink = math.prod((1 - lr * WEIGHT_DECAY) ** steps_per_epoch for lr in history["lr"])
print(f"Seed 0: weight decay alone multiplied the weights by {shrink:.4f} over the whole run.")

**Reading the log and the plots for seed 0** (the red line marks the restored epoch). The training loss falls steadily, while the validation loss improves only slowly after epoch 2 and reaches its best value, 0.396, at epoch 7. The growing gap between the two curves is overfitting. Epochs 8 to 10 brought no improvement, so the plateau scheduler halved the learning rate. Epoch 11 then reached 0.395, but that is less than 0.001 below the best value, so by our definition of improvement it does not count. After six epochs without an improvement, early stopping ended the run after epoch 13, and the weights of epoch 7 were restored.

Four observations from the three seeds:
- **The seed alone changes the result noticeably.** The runs stopped after 13, 16 and 14 epochs and reached test accuracies between 85.2 % and 86.7 %. A difference of this size between two *methods* therefore means nothing unless it holds across several seeds.
- **The test loss is higher than the validation loss, for every seed, by 0.03 to 0.04.** Two things can cause such a gap. Choosing the best of many noisy validation values makes that value optimistic. And the official test split need not be exactly as hard as our validation subset. In a separate check without any epoch selection we found a gap of the same size, so here it is mainly the second. Either way, the number to report is the one from the test set, which played no part in any decision.
- **Weight decay played almost no role here.** The last printed line shows its total effect: the weights were shrunk by less than 1 % over the whole run, because $\eta\lambda = 10^{-5}$ per step and the run is short. Weight decay matters in long runs with many steps.
- **Early stopping is a safeguard, not a performance booster.** The fixed cosine runs of §5.6 reached a lower validation loss (about 0.36) in 10 epochs than any of the three restored models (0.38 to 0.40). Early stopping is most useful when you do not know in advance how long to train.

## 7.5 Loading the best model from disk
`torch.load(..., weights_only=True)` accepts only tensors and basic Python types, which makes it the safe way to load files. It has been the default since PyTorch 2.6.

In [ ]:
checkpoint = torch.load("best_model_seed0.pt", map_location=device, weights_only=True)
fresh = make_mlp()
fresh.load_state_dict(checkpoint["model_state"])
print(f"checkpoint from epoch {checkpoint['epoch']}: validation loss when saved {checkpoint['value']:.4f}, "
      f"after reloading {evaluate(fresh, val_loader)[0]:.4f}")

💡 The file above is enough to *use* the best model. To **resume** an interrupted run, a checkpoint must also contain `optimizer.state_dict()` (the momentum or Adam averages), `scheduler.state_dict()`, the epoch number and the early-stopping counters.

## 7.6 The most effective remedy: more data
Everything in §7 manages the overfitting caused by training on only 10,000 images. The setup held back 45,000 more training images. The cell trains the same MLP with the cosine schedule of §5.6 on all 55,000 (every training image except the 5,000 validation images) and compares it with the 10,000-image run of §5.6 on the same validation set.

In [ ]:
rest_idx = perm[15_000:]                                               # the 45,000 training images not used so far
x_rest = (as_vectors(raw_train.data[rest_idx]) - mean) / std           # same normalization as before
big_loader = DataLoader(TensorDataset(torch.cat([x_train, x_rest]), torch.cat([y_train, raw_train.targets[rest_idx]])),
                        batch_size=128, shuffle=True, generator=torch.Generator().manual_seed(0))

torch.manual_seed(0)
model = make_mlp()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=10 * len(big_loader))     # stepped per batch
for _ in range(10):
    train_one_epoch(model, big_loader, optimizer, scheduler)
big_loss, big_acc = evaluate(model, val_loader)
small_loss, small_acc = schedule_runs[("cosine", 0)][0][-1]            # the 10-epoch cosine run of §5.6, seed 0
print(f"10,000 training images: validation loss {small_loss:.4f}, accuracy {small_acc:.4f}")
print(f"55,000 training images: validation loss {big_loss:.4f}, accuracy {big_acc:.4f}")

With 5.5 times as many images, the validation accuracy rises by about four percentage points and the validation loss falls by about a quarter. No choice of initialization, optimizer setting, schedule or stopping rule in this notebook comes close to that. (More images also mean more update steps in the same 10 epochs; both come with more data.) When a model overfits, getting more data, or augmenting the data you have, is the first thing to consider.

**Takeaway.** Early stopping = validation loss + a CPU copy of the best weights (also on disk) + patience clearly larger than the scheduler's + restore at the end. The test set is used once per finished model, and results are reported over several seeds.

---
# Summary

| Topic | Remember |
| --- | --- |
| Tensor internals | storage + offset + shape + strides; views alias memory; `view` needs a compatible layout, `reshape` copies if needed, `clone` always copies. |
| `einsum` | letters shared between operands are multiplied, letters missing after `->` are summed. |
| Initialization | decisive for deep networks without normalization (He for ReLU, Xavier for tanh/sigmoid), unimportant for shallow or normalized ones; Adam can mask a poor scale; output layer small; initial loss ≈ ln(number of classes). |
| Optimizers | AdamW (`lr=1e-3`, `weight_decay=0.01`) first; SGD + momentum with a schedule for established CNN recipes; compare optimizers only with equal tuning effort. |
| Learning rate | a short log-scale sweep on the validation set, then confirm with a full run. |
| Schedulers | create after the optimizer, `scheduler.step()` after `optimizer.step()`, same unit for sizing and stepping, look at the learning rate; cosine for a fixed budget, plus warm-up where the first steps are unstable; ReduceLROnPlateau for an unknown length. |
| Weight decay | `AdamW(weight_decay=λ)`, not `Adam(weight_decay=λ)`; skip biases and normalization parameters. |
| Gradient clipping | `clip_grad_norm_` between `backward()` and `step()`; log the returned norm. |
| Early stopping | validation loss, `eval()` + `no_grad`, a copy of the best `state_dict`, patience larger than the scheduler's, restore the best weights; report the test set, once per finished model. |
| Evidence | measure on your own problem, with several seeds: here more data, the learning rate and its decay mattered; initialization mattered only for the deep network trained with SGD; warm-up and weight decay did not matter. |

## Further reading
- PyTorch documentation: [tensor views](https://docs.pytorch.org/docs/stable/tensor_view.html), [`torch.einsum`](https://docs.pytorch.org/docs/stable/generated/torch.einsum.html), [`torch.nn.init`](https://docs.pytorch.org/docs/stable/nn.init.html), [`torch.optim` and schedulers](https://docs.pytorch.org/docs/stable/optim.html), [saving and loading](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html).
- K. He et al., *Delving Deep into Rectifiers*, ICCV 2015 (He initialization).
- D. P. Kingma and J. Ba, *Adam: A Method for Stochastic Optimization*, ICLR 2015.
- I. Loshchilov and F. Hutter, *Decoupled Weight Decay Regularization*, ICLR 2019 (AdamW).